<a href="https://colab.research.google.com/github/Grenki-with-cheese/dissertation-notebook-2213935-cn6000/blob/main/notebook06_Recovered_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setting up SD v1.4 with Erased checkpoint (see notebook04, notebook04)

In [ ]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
print("Install complete.")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime (A100 or L4)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
from pathlib import Path
from diffusers import StableDiffusionPipeline
import safetensors.torch

# Use the path to your ESD checkpoint
CHECKPOINT_PATH = Path("checkpoints/esd_vangogh_2026-05-06.safetensors")
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

state = safetensors.torch.load_file(str(CHECKPOINT_PATH))
result = pipe.unet.load_state_dict(state, strict=False)
print(f"Loaded {len(state)} cross-attention keys.")
print(f"Missing keys (frozen layers, expected): {len(result.missing_keys)}")
assert len(result.unexpected_keys) == 0, f"Unexpected keys: {result.unexpected_keys[:5]}"

Generating using recovered prompts (from notebook05)

In [4]:
import json

SEEDS = [42, 1337, 2026, 7777, 2213935]
GUIDANCE = 9.0
STEPS = 50
NEGATIVE = ("photograph, photo, realistic, 3d render, cgi, blurry, deformed face, "
            "extra fingers, ugly, low quality, oversaturated, anime, cartoon, "
            "watermark, signature, text")

def generate_set(prompts, seeds, output_dir):
    """Generate one image per (prompt, seed) pair, saved with paired filename."""
    output_dir.mkdir(parents=True, exist_ok=True)
    log = []
    total = len(prompts) * len(seeds)
    count = 0
    for prompt_idx, prompt in enumerate(prompts):
        for seed in seeds:
            generator = torch.Generator("cuda").manual_seed(seed)
            image = pipe(
                prompt,
                negative_prompt=NEGATIVE,
                generator=generator,
                num_inference_steps=STEPS,
                guidance_scale=GUIDANCE,
            ).images[0]
            filename = f"p{prompt_idx:03d}_s{seed:06d}.png"
            image.save(output_dir / filename)
            log.append({
                "prompt_idx": prompt_idx,
                "prompt": prompt,
                "seed": seed,
                "filename": filename,
            })
            count += 1
            if count % 10 == 0 or count == total:
                print(f"  {count}/{total}")
    return log

recovered = json.loads(Path("prompts/recovered_prompts.json").read_text())
assert len(recovered) == 10, f"Expected 10 recovered prompts, got {len(recovered)}"

print(f"Generating {len(recovered) * len(SEEDS)} recovered images...")
log_r = generate_set(recovered, SEEDS, Path("outputs/recovered"))

Path("logs").mkdir(exist_ok=True)
Path("logs/seeds_recovered.json").write_text(
    json.dumps({"recovered": log_r}, indent=2)
)
print(f"\nSaved seed log to logs/seeds_recovered.json")

Generating 50 recovered images...
  10/50
  20/50
  30/50
  40/50
  50/50

Saved seed log to logs/seeds_recovered.json


Verifying image count

In [5]:
n_recovered = len(list(Path("outputs/recovered").glob("*.png")))
assert n_recovered == 50, f"Recovered count mismatch: expected 50, got {n_recovered}"
print(f"Recovered generation complete: {n_recovered} images")

Recovered generation complete: 50 images
